# Fraud Detection Baseline Model

This notebook builds a transaction-only Logistic Regression baseline using chronological validation.

In [4]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

import sys

print("Python executable:", sys.executable)
print("Pandas version:", pd.__version__)

Python executable: d:\ML\New Project Folder 26.07.2026\1st project\fraud-detection-platform\.venv312\Scripts\python.exe
Pandas version: 3.0.5


In [5]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

TRANSACTION_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "train_transaction.csv"
)

print("Project root:", PROJECT_ROOT)
print("Dataset exists:", TRANSACTION_PATH.exists())

Project root: d:\ML\New Project Folder 26.07.2026\1st project\fraud-detection-platform
Dataset exists: True


In [6]:
selected_columns = [
    "TransactionID",
    "TransactionDT",
    "TransactionAmt",
    "ProductCD",
    "card1",
    "card2",
    "card3",
    "card4",
    "card5",
    "card6",
    "addr1",
    "addr2",
    "dist1",
    "P_emaildomain",
    "C1",
    "C2",
    "C4",
    "C5",
    "C6",
    "C8",
    "C10",
    "C11",
    "C12",
    "C13",
    "C14",
    "isFraud",
]

model_df = pd.read_csv(
    TRANSACTION_PATH,
    usecols=selected_columns,
    low_memory=False,
)

print("Dataset shape:", model_df.shape)

model_df.head()

Dataset shape: (590540, 26)


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,C2,C4,C5,C6,C8,C10,C11,C12,C13,C14
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,1.0,0.0,0.0,1.0,0.0,0.0,2.0,0.0,1.0,1.0
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,5.0,0.0,0.0,4.0,0.0,0.0,1.0,0.0,25.0,1.0
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0


In [7]:
model_df = model_df.sort_values(
    "TransactionDT"
).reset_index(drop=True)

model_df["transaction_hour"] = (
    model_df["TransactionDT"] // 3600
) % 24

model_df["transaction_day"] = (
    model_df["TransactionDT"] // 86400
)

model_df["transaction_amount_log"] = np.log1p(
    model_df["TransactionAmt"]
)

print(model_df.shape)

model_df[
    [
        "TransactionDT",
        "transaction_hour",
        "transaction_day",
        "TransactionAmt",
        "transaction_amount_log",
    ]
].head()

(590540, 29)


,TransactionDT,transaction_hour,transaction_day,TransactionAmt,transaction_amount_log
0,86400,0,1,68.5,4.241327
1,86401,0,1,29.0,3.401197
2,86469,0,1,59.0,4.094345
3,86499,0,1,50.0,3.931826
4,86506,0,1,50.0,3.931826


In [8]:
total_rows = len(model_df)

train_end = int(total_rows * 0.70)
validation_end = int(total_rows * 0.85)

train_df = model_df.iloc[:train_end].copy()

validation_df = model_df.iloc[
    train_end:validation_end
].copy()

test_df = model_df.iloc[
    validation_end:
].copy()

print("Training rows:", len(train_df))
print("Validation rows:", len(validation_df))
print("Testing rows:", len(test_df))

Training rows: 413378
Validation rows: 88581
Testing rows: 88581


In [9]:
split_summary = pd.DataFrame({
    "split": [
        "Training",
        "Validation",
        "Testing",
    ],
    "rows": [
        len(train_df),
        len(validation_df),
        len(test_df),
    ],
    "fraud_rate_percent": [
        train_df["isFraud"].mean() * 100,
        validation_df["isFraud"].mean() * 100,
        test_df["isFraud"].mean() * 100,
    ],
})

split_summary

,split,rows,fraud_rate_percent
0,Training,413378,3.516878
1,Validation,88581,3.434145
2,Testing,88581,3.480430


In [10]:
target_column = "isFraud"

excluded_columns = [
    "TransactionID",
    "isFraud",
]

feature_columns = [
    column
    for column in model_df.columns
    if column not in excluded_columns
]

X_train = train_df[feature_columns]
y_train = train_df[target_column]

X_validation = validation_df[feature_columns]
y_validation = validation_df[target_column]

X_test = test_df[feature_columns]
y_test = test_df[target_column]

print("Training features:", X_train.shape)
print("Validation features:", X_validation.shape)
print("Testing features:", X_test.shape)

Training features: (413378, 27)
Validation features: (88581, 27)
Testing features: (88581, 27)


In [11]:
categorical_features = [
    "ProductCD",
    "card4",
    "card6",
    "P_emaildomain",
]

numerical_features = [
    column
    for column in feature_columns
    if column not in categorical_features
]

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

print("\nCategorical feature names:")
print(categorical_features)

Numerical features: 23
Categorical features: 4

Categorical feature names:
['ProductCD', 'card4', 'card6', 'P_emaildomain']


# Step 10: Create preprocessing pipelines

Numerical missing values will be filled using the median.

Categorical missing values will be filled using the most common value and converted into numbers through one-hot encoding.

In [12]:
numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=50,
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            numerical_features,
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features,
        ),
    ]
)

# Step 11: Create the Logistic Regression model

In [13]:

baseline_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor,
        ),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=500,
                solver="saga",
                random_state=42,
            ),
        ),
    ]
)

baseline_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float

In [14]:
class_weight="balanced"

In [15]:
baseline_model.fit(
    X_train,
    y_train,
)

d:\ML\New Project Folder 26.07.2026\1st project\fraud-detection-platform\.venv312\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](27,)","['TransactionDT','TransactionAmt','ProductCD',...,'transaction_hour', 'transaction_day','transaction_amount_log']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,27
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default o

In [16]:
validation_probabilities = baseline_model.predict_proba(
    X_validation
)[:, 1]

validation_predictions = (
    validation_probabilities >= 0.50
).astype(int)

In [17]:
validation_pr_auc = average_precision_score(
    y_validation,
    validation_probabilities,
)

validation_roc_auc = roc_auc_score(
    y_validation,
    validation_probabilities,
)

print(
    f"Validation PR-AUC: "
    f"{validation_pr_auc:.4f}"
)

print(
    f"Validation ROC-AUC: "
    f"{validation_roc_auc:.4f}"
)

Validation PR-AUC: 0.0370
Validation ROC-AUC: 0.5225


In [18]:
print(
    classification_report(
        y_validation,
        validation_predictions,
        target_names=[
            "Legitimate",
            "Fraud",
        ],
        digits=4,
        zero_division=0,
    )
)

              precision    recall  f1-score   support

  Legitimate     0.0000    0.0000    0.0000     85539
       Fraud     0.0343    1.0000    0.0664      3042

    accuracy                         0.0343     88581
   macro avg     0.0172    0.5000    0.0332     88581
weighted avg     0.0012    0.0343    0.0023     88581



In [19]:
confusion_matrix_values = confusion_matrix(
    y_validation,
    validation_predictions,
)

confusion_matrix_table = pd.DataFrame(
    confusion_matrix_values,
    index=[
        "Actual Legitimate",
        "Actual Fraud",
    ],
    columns=[
        "Predicted Legitimate",
        "Predicted Fraud",
    ],
)

confusion_matrix_table

,Predicted Legitimate,Predicted Fraud
Actual Legitimate,0,85539
Actual Fraud,0,3042


In [20]:
probability_summary = pd.Series(
    validation_probabilities,
    name="fraud_probability",
).describe(
    percentiles=[
        0.01,
        0.05,
        0.25,
        0.50,
        0.75,
        0.95,
        0.99,
    ]
)

print(probability_summary)

print(
    "\nPercentage predicted as fraud at threshold 0.50:",
    round(
        (validation_probabilities >= 0.50).mean() * 100,
        2,
    ),
    "%",
)

count    88581.000000
mean         0.602684
std          0.031863
min          0.530554
1%           0.540411
5%           0.551120
25%          0.577273
50%          0.602391
75%          0.627870
95%          0.654733
99%          0.664948
max          0.700072
Name: fraud_probability, dtype: float64

Percentage predicted as fraud at threshold 0.50: 100.0 %


## Corrected Scaled Logistic Regression Baseline

The first model did not converge and predicted every validation transaction as fraud. This version scales numerical features and improves missing-value handling.

Import StandardScaler

In [21]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler

Create improved preprocessing

In [22]:
numerical_pipeline_scaled = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
            ),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ]
)

categorical_pipeline_scaled = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="__MISSING__",
            ),
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=50,
            ),
        ),
    ]
)

preprocessor_scaled = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline_scaled,
            numerical_features,
        ),
        (
            "categorical",
            categorical_pipeline_scaled,
            categorical_features,
        ),
    ]
)

Create the corrected model

In [23]:
scaled_baseline_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor_scaled,
        ),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                solver="lbfgs",
                max_iter=1000,
                random_state=42,
            ),
        ),
    ]
)

scaled_baseline_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float

Train it

In [24]:
scaled_baseline_model.fit(
    X_train,
    y_train,
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](27,)","['TransactionDT','TransactionAmt','ProductCD',...,'transaction_hour', 'transaction_day','transaction_amount_log']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,27
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default o

Generate validation probabilities

In [25]:
scaled_validation_probabilities = (
    scaled_baseline_model.predict_proba(X_validation)[:, 1]
)

scaled_validation_predictions = (
    scaled_validation_probabilities >= 0.50
).astype(int)

Check the probability distribution

In [26]:
probability_summary_scaled = pd.Series(
    scaled_validation_probabilities,
    name="fraud_probability",
).describe(
    percentiles=[
        0.01,
        0.05,
        0.25,
        0.50,
        0.75,
        0.95,
        0.99,
    ]
)

print(probability_summary_scaled)

print(
    "\nPercentage predicted as fraud:",
    round(scaled_validation_predictions.mean() * 100, 2),
    "%",
)

count    88581.000000
mean         0.465339
std          0.189704
min          0.000185
1%           0.021205
5%           0.155052
25%          0.347550
50%          0.442465
75%          0.578784
95%          0.812149
99%          0.921321
max          0.999804
Name: fraud_probability, dtype: float64

Percentage predicted as fraud: 37.34 %


## Calculate PR-AUC and ROC-AUC

In [27]:
scaled_pr_auc = average_precision_score(
    y_validation,
    scaled_validation_probabilities,
)

scaled_roc_auc = roc_auc_score(
    y_validation,
    scaled_validation_probabilities,
)

print(f"Scaled Logistic Regression PR-AUC: {scaled_pr_auc:.4f}")
print(f"Scaled Logistic Regression ROC-AUC: {scaled_roc_auc:.4f}")

Scaled Logistic Regression PR-AUC: 0.2705
Scaled Logistic Regression ROC-AUC: 0.7745


## Generate the classification report

In [28]:
print(
    classification_report(
        y_validation,
        scaled_validation_predictions,
        target_names=[
            "Legitimate",
            "Fraud",
        ],
        digits=4,
        zero_division=0,
    )
)

              precision    recall  f1-score   support

  Legitimate     0.9863    0.6400    0.7763     85539
       Fraud     0.0690    0.7505    0.1264      3042

    accuracy                         0.6438     88581
   macro avg     0.5277    0.6952    0.4513     88581
weighted avg     0.9548    0.6438    0.7539     88581



## Generate the confusion matrix

In [29]:
scaled_confusion_matrix = confusion_matrix(
    y_validation,
    scaled_validation_predictions,
)

scaled_confusion_matrix_table = pd.DataFrame(
    scaled_confusion_matrix,
    index=[
        "Actual Legitimate",
        "Actual Fraud",
    ],
    columns=[
        "Predicted Legitimate",
        "Predicted Fraud",
    ],
)

scaled_confusion_matrix_table

,Predicted Legitimate,Predicted Fraud
Actual Legitimate,54742,30797
Actual Fraud,759,2283


## Validation Threshold Analysis

The default threshold of 0.50 detects many fraudulent transactions but produces too many false-positive alerts. This section compares different thresholds using the validation dataset.

In [30]:
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
)

thresholds = np.arange(
    0.30,
    0.96,
    0.05,
)

threshold_results = []

for threshold in thresholds:
    predictions = (
        scaled_validation_probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_validation,
        predictions,
    ).ravel()

    threshold_results.append({
        "threshold": round(threshold, 2),
        "fraud_precision": precision_score(
            y_validation,
            predictions,
            zero_division=0,
        ),
        "fraud_recall": recall_score(
            y_validation,
            predictions,
            zero_division=0,
        ),
        "fraud_f1": f1_score(
            y_validation,
            predictions,
            zero_division=0,
        ),
        "transactions_flagged_percent": (
            predictions.mean() * 100
        ),
        "false_positive_rate": (
            fp / (fp + tn)
        ),
        "true_negatives": tn,
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp,
    })

threshold_results_df = pd.DataFrame(
    threshold_results
)

threshold_results_df.round(4)

,threshold,fraud_precision,fraud_recall,fraud_f1,transactions_flagged_percent,false_positive_rate,true_negatives,false_positives,false_negatives,true_positives
0,0.30,0.0390,0.9599,0.0749,84.6299,0.8423,13493,72046,122,2920
1,0.35,0.0427,0.9250,0.0816,74.3929,0.7375,22455,63084,228,2814
2,0.40,0.0492,0.8771,0.0931,61.2366,0.6030,33963,51576,374,2668
3,0.45,0.0578,0.8123,0.1079,48.2508,0.4708,45269,40270,571,2471
4,0.50,0.0690,0.7505,0.1264,37.3444,0.3600,54742,30797,759,2283
5,0.55,0.0824,0.6940,0.1473,28.9272,0.2749,62026,23513,931,2111
6,0.60,0.0964,0.6249,0.1671,22.2508,0.2082,67730,17809,1141,1901
7,0.65,0.1135,0.5648,0.1890,17.0940,0.1569,72115,13424,1324,1718
8,0.70,0.1309,0.4951,0.2070,12.9926,0.1169,75536,10003,1536,1506
9,0.75,0.1642,0.4356,0.2385,9.1069,0.0788,78797,6742,1717,1325


## 1. Automatically select the best F1 threshold

Add this cell:

In [31]:
best_threshold_row = threshold_results_df.loc[
    threshold_results_df["fraud_f1"].idxmax()
]

selected_threshold = float(
    best_threshold_row["threshold"]
)

print("Selected threshold:", selected_threshold)
print()
print(best_threshold_row.round(4))

Selected threshold: 0.85

threshold                           0.8500
fraud_precision                     0.3147
fraud_recall                        0.3037
fraud_f1                            0.3091
transactions_flagged_percent        3.3145
false_positive_rate                 0.0235
true_negatives                  83527.0000
false_positives                  2012.0000
false_negatives                  2118.0000
true_positives                    924.0000
Name: 11, dtype: float64


## Generate test probabilities

In [32]:
test_probabilities = (
    scaled_baseline_model.predict_proba(X_test)[:, 1]
)

test_predictions = (
    test_probabilities >= selected_threshold
).astype(int)

print(
    "Test transactions flagged:",
    round(test_predictions.mean() * 100, 2),
    "%",
)

Test transactions flagged: 5.79 %


## Calculate test PR-AUC and ROC-AUC

In [33]:
test_pr_auc = average_precision_score(
    y_test,
    test_probabilities,
)

test_roc_auc = roc_auc_score(
    y_test,
    test_probabilities,
)

print(
    f"Test PR-AUC: {test_pr_auc:.4f}"
)

print(
    f"Test ROC-AUC: {test_roc_auc:.4f}"
)

Test PR-AUC: 0.1874
Test ROC-AUC: 0.7683


In [34]:
print(
    classification_report(
        y_test,
        test_predictions,
        target_names=[
            "Legitimate",
            "Fraud",
        ],
        digits=4,
        zero_division=0,
    )
)

              precision    recall  f1-score   support

  Legitimate     0.9738    0.9505    0.9620     85498
       Fraud     0.1753    0.2916    0.2189      3083

    accuracy                         0.9276     88581
   macro avg     0.5746    0.6211    0.5905     88581
weighted avg     0.9460    0.9276    0.9362     88581



## Generate the test confusion matrix

In [35]:
test_confusion_matrix = confusion_matrix(
    y_test,
    test_predictions,
)

test_confusion_matrix_table = pd.DataFrame(
    test_confusion_matrix,
    index=[
        "Actual Legitimate",
        "Actual Fraud",
    ],
    columns=[
        "Predicted Legitimate",
        "Predicted Fraud",
    ],
)

test_confusion_matrix_table

,Predicted Legitimate,Predicted Fraud
Actual Legitimate,81268,4230
Actual Fraud,2184,899


## Final Baseline Results

The scaled Logistic Regression model achieved:

### Validation
- PR-AUC: 0.2705
- ROC-AUC: 0.7745
- Selected threshold: 0.85
- Fraud precision: 31.47%
- Fraud recall: 30.37%
- Transactions flagged: 3.31%

### Test
- PR-AUC: 0.1874
- ROC-AUC: 0.7683
- Fraud precision: 17.53%
- Fraud recall: 29.16%
- Transactions flagged: 5.79%

The model detected 899 of 3,083 fraud transactions in the test period but generated 4,230 false-positive alerts.

Logistic Regression provides a useful benchmark, but its nonlinear modelling capacity and test precision are insufficient for the final fraud-detection system. The next experiment will use gradient-boosted decision trees.